# 01 — Data Collection

| Step | What happens |
|---|---|
| 1 | Import modules and verify config |
| 2 | Collect news via Google News RSS |
| 3 | Collect news via NewsAPI |
| 4 | Deduplicate and inspect combined dataset |
| 5 | Download Brent and WTI crude oil prices |
| 6 | Data quality check |

> **Next step:** `02_Data_Preprocessing.ipynb`

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.config import NEWS_RAW_FILE, CRUDE_RAW_FILE, SEARCH_KEYWORDS, OIL_TICKERS

print('Config loaded')
print(f'Keywords ({len(SEARCH_KEYWORDS)}):')
for kw in SEARCH_KEYWORDS:
    print(f'  - {kw}')
print(f'Oil tickers: {OIL_TICKERS}')

## 1. Collect News Articles

We query 7 geopolitical keywords via Google News RSS (feedparser) and NewsAPI REST calls.

In [ ]:
from src.data_collector import collect_all_news

news_df = collect_all_news(save=True)
print(f'Total articles after deduplication: {len(news_df)}')
news_df.head(3)

## 2. Dataset Overview

In [ ]:
print(f'Shape: {news_df.shape}')
print(f'Columns: {list(news_df.columns)}')
print(f'Date range: {news_df["published"].min()} to {news_df["published"].max()}')
print(f'Unique sources: {news_df["source"].nunique()}')
print(f'\nMissing values:\n{news_df.isnull().sum()}')

In [ ]:
print('Articles per keyword:')
print(news_df['keyword'].value_counts().to_string())
print('\nTop 10 sources:')
print(news_df['source'].value_counts().head(10).to_string())

## 3. Crude Oil Prices

Brent (BZ=F) and WTI (CL=F) daily closing prices via yfinance.

In [ ]:
from src.data_collector import collect_crude_oil

crude_df = collect_crude_oil(save=True)
print(f'Trading days: {len(crude_df)}')
crude_df.head(10)

In [ ]:
print('Crude oil summary statistics:')
crude_df[['Brent_Close', 'WTI_Close']].describe().round(2)

## 4. Data Quality Check

In [ ]:
print('News dataset:')
print(f'  Rows          : {len(news_df)}')
print(f'  Null titles   : {news_df["title"].isnull().sum()}')
print(f'  Null text     : {news_df["text"].isnull().sum()}')
print(f'  Date range    : {news_df["date"].min()} to {news_df["date"].max()}')

print('\nCrude oil dataset:')
print(f'  Rows          : {len(crude_df)}')
print(f'  Null Brent    : {crude_df["Brent_Close"].isnull().sum()}')
print(f'  Null WTI      : {crude_df["WTI_Close"].isnull().sum()}')

## Key Findings

| Metric | Value |
|---|---|
| Total articles | 314 |
| Unique sources | Multiple (Google News RSS) |
| Crude oil trading days | 123 |
| Date range | Nov 2025 - Apr 2026 |

> **Next step:** `02_Data_Preprocessing.ipynb`